# 8-Puzzle con A*

## Heurística utilizada

Además de calcular la distancia Manhattan de cada ficha, puede entenderse la heurística de la siguiente manera:

> Cada ficha "genera una presión" sobre el tablero dependiendo de qué tan lejos está de su posición final. La suma de esas presiones representa el esfuerzo mínimo que todavía falta por realizar. Aunque varias fichas puedan moverse al mismo tiempo durante la búsqueda, ninguna puede llegar a su destino sin recorrer al menos esa distancia, por lo que esta estimación sirve como una guía para priorizar los estados más prometedores sin sobreestimar el costo real.

Esta explicación es una interpretación intuitiva del funcionamiento de la distancia Manhattan y ayuda a comprender por qué A* explora primero los estados que parecen acercarse más a la solución.


In [ ]:
import heapq

def leer_estado(nombre):
    with open(nombre) as f:
        return tuple(tuple(map(int,line.strip())) for line in f)

inicio=leer_estado("estadoinicial_taller1.txt")
meta=leer_estado("estadofinal_taller1.txt")

objetivo={}
for i in range(3):
    for j in range(3):
        objetivo[meta[i][j]]=(i,j)

def vacio(e):
    for i in range(3):
        for j in range(3):
            if e[i][j]==0:
                return i,j

def h(e):
    s=0
    for i in range(3):
        for j in range(3):
            v=e[i][j]
            if v:
                x,y=objetivo[v]
                s+=abs(i-x)+abs(j-y)
    return s

def vecinos(e):
    f,c=vacio(e)
    datos=[]
    for df,dc,nombre in [(-1,0,"Arriba"),(1,0,"Abajo"),(0,-1,"Izquierda"),(0,1,"Derecha")]:
        nf,nc=f+df,c+dc
        if 0<=nf<3 and 0<=nc<3:
            n=[list(r) for r in e]
            n[f][c],n[nf][nc]=n[nf][nc],n[f][c]
            datos.append((tuple(tuple(r) for r in n),nombre))
    return datos

def astar(i,m):
    frontera=[(h(i),0,i)]
    padre={i:None}
    mov={i:None}
    costo={i:0}
    iteraciones=0
    while frontera:
        _,g,act=heapq.heappop(frontera)
        iteraciones+=1
        if act==m:
            estados=[]
            movimientos=[]
            while act is not None:
                estados.append(act)
                if mov[act] is not None:
                    movimientos.append(mov[act])
                act=padre[act]
            return estados[::-1],movimientos[::-1],iteraciones
        for vec,movimiento in vecinos(act):
            ng=g+1
            if vec not in costo or ng<costo[vec]:
                costo[vec]=ng
                padre[vec]=act
                mov[vec]=movimiento
                heapq.heappush(frontera,(ng+h(vec),ng,vec))
    return None,None,iteraciones

camino,movs,it=astar(inicio,meta)

if camino is None:
    print("No existe solución.")
else:
    print("Iteraciones realizadas:",it)
    print("Movimientos necesarios:",len(movs))
    print("\nSecuencia de movimientos:")
    for k,m in enumerate(movs,1):
        print(f"{k}. {m}")
    print("\nEstados:")
    for p,e in enumerate(camino):
        print(f"\nPaso {p}")
        for fila in e:
            print(*fila)
